# Coloriser des photos noir et blanc avec un CNN

Un réseau encodeur-décodeur reçoit une image en noir et blanc et prédit ses couleurs.
Données : CIFAR-10 (60 000 images 32×32). Deux modèles comparés :

1. un **autoencodeur convolutionnel** minuscule (446 paramètres) ;
2. un **U-Net** (un encodeur-décodeur avec des raccourcis entre les deux).

L'astuce qui rend le problème simple : l'espace de couleur **Lab**. L (la luminosité) = l'image noir et blanc,
a et b = la couleur. Le réseau reçoit L et ne prédit que a et b.

Tourne sur CPU (lent) ou GPU (Google Colab : Exécution → Modifier le type d'exécution → GPU).

## 0. Imports et réglages

In [ ]:
import os, time, copy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import torchvision

from skimage.color import rgb2lab, lab2rgb

In [ ]:
SEED = 456
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 128
EPOCHS = 30          # maximum : l'early stopping s'arrête en général avant
LR = 2e-3
PATIENCE = 5         # early stopping : époques sans progrès en validation avant d'arrêter
LR_PATIENCE = 2      # scheduler : époques sans progrès avant de baisser le learning rate
LR_FACTOR = 0.5

os.makedirs("models", exist_ok=True)
print(DEVICE)

## 1. Les données : CIFAR-10

torchvision télécharge le dataset en une ligne. On fusionne train et test (60 000 images),
on passe en tenseur (N, C, H, W) entre 0 et 1, puis on découpe 80 / 10 / 10.

In [ ]:
train_set = torchvision.datasets.CIFAR10(root="./data", train=True, download=True)
test_set = torchvision.datasets.CIFAR10(root="./data", train=False, download=True)
x = np.concatenate([train_set.data, test_set.data])      # (60000, 32, 32, 3), uint8
x = torch.from_numpy(x).permute(0, 3, 1, 2).float() / 255  # -> (N, 3, 32, 32) dans [0, 1]

idx = torch.randperm(len(x), generator=torch.Generator().manual_seed(SEED))
n_train, n_val = int(0.8 * len(x)), int(0.1 * len(x))
x_train = x[idx[:n_train]]
x_val = x[idx[n_train:n_train + n_val]]
x_test = x[idx[n_train + n_val:]]
print("train", tuple(x_train.shape), "| val", tuple(x_val.shape), "| test", tuple(x_test.shape))

## 2. Entraînement, évaluation, affichage

In [ ]:
def evaluate(model, loader, loss_fn=nn.MSELoss()):
    """Erreur moyenne sur un loader (validation ou test)."""
    model.eval()
    total, n = 0.0, 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            total += loss_fn(model(xb), yb).item() * len(xb)
            n += len(xb)
    return total / n


def train_model(model, train_loader, val_loader, epochs=EPOCHS, lr=LR, loss_fn=nn.MSELoss()):
    """Adam + MSE, learning rate qui baisse quand la validation stagne, early stopping."""
    model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=LR_FACTOR, patience=LR_PATIENCE)
    history = {"train_loss": [], "val_loss": []}
    best_loss, best_state, best_epoch, bad_epochs = float("inf"), None, 0, 0

    for epoch in range(1, epochs + 1):
        t0 = time.time()
        model.train()
        total, n = 0.0, 0
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            optimizer.step()
            total += loss.item() * len(xb)
            n += len(xb)
        val_loss = evaluate(model, val_loader, loss_fn)
        history["train_loss"].append(total / n)
        history["val_loss"].append(val_loss)
        print(f"epoch {epoch:2d} | train {total / n:.5f} | val {val_loss:.5f} | {time.time() - t0:.1f}s")
        scheduler.step(val_loss)

        if val_loss < best_loss:
            best_loss, best_state, best_epoch, bad_epochs = val_loss, copy.deepcopy(model.state_dict()), epoch, 0
        else:
            bad_epochs += 1
            if bad_epochs >= PATIENCE:
                print(f"early stopping (meilleure époque : {best_epoch})")
                break

    model.load_state_dict(best_state)
    history["best_epoch"] = best_epoch
    return history


def predict(model, x, batch_size=500):
    model.eval()
    with torch.no_grad():
        return torch.cat([model(x[i:i + batch_size].to(DEVICE)).cpu() for i in range(0, len(x), batch_size)])


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def plot_learning_curve(history, title=""):
    epochs = range(1, len(history["train_loss"]) + 1)
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(epochs, history["train_loss"], label="train")
    ax.plot(epochs, history["val_loss"], label="validation")
    ax.axvline(history["best_epoch"], color="gray", ls="--", lw=1, label="meilleure époque")
    ax.set_xlabel("époque"); ax.set_ylabel("MSE"); ax.set_title(title); ax.legend()
    plt.tight_layout(); plt.show()


def show_rows(rows, n=10, title=""):
    """rows : dict nom -> tenseur d'images (1 ou 3 canaux)."""
    fig, axes = plt.subplots(len(rows), n, figsize=(1.5 * n, 1.6 * len(rows)))
    for r, (label, imgs) in enumerate(rows.items()):
        for i in range(n):
            img = imgs[i].clamp(0, 1)
            if img.shape[0] == 1:
                axes[r, i].imshow(img[0].numpy(), cmap="gray", vmin=0, vmax=1)
            else:
                axes[r, i].imshow(img.permute(1, 2, 0).numpy())
            axes[r, i].axis("off")
        axes[r, 0].set_title(label, loc="left", fontsize=9)
    fig.suptitle(title); plt.tight_layout(); plt.show()

## 3. L'astuce : l'espace Lab

**L** (0 à 100) est la luminosité, c'est-à-dire la photo noir et blanc. **a** et **b** (à peu près -128 à 127) sont la couleur.
On remet L dans [0, 1] et ab dans [-1, 1] (d'où le `tanh` en sortie des modèles).

Pour juger les modèles, on compare aussi à une référence bête : recopier le noir et blanc sur les 3 canaux.
Et on mesure la **chroma** moyenne, √(a² + b²) : plus elle est haute, plus les couleurs sont franches.

In [ ]:
def rgb_to_lab(x, chunk=5000):
    """(N, 3, H, W) RGB dans [0, 1] -> L dans [0, 1] (N, 1, H, W) et ab dans [-1, 1] (N, 2, H, W)."""
    L, ab = [], []
    for i in range(0, len(x), chunk):   # par paquets : rgb2lab calcule en float64 et prend beaucoup de mémoire
        lab = torch.from_numpy(rgb2lab(x[i:i + chunk].permute(0, 2, 3, 1).numpy())).float().permute(0, 3, 1, 2)
        L.append(lab[:, :1] / 100)
        ab.append(lab[:, 1:] / 128)
    return torch.cat(L), torch.cat(ab)


def lab_to_rgb(L, ab):
    lab = torch.cat([L * 100, ab * 128], dim=1).permute(0, 2, 3, 1).numpy().astype(np.float64)
    return torch.from_numpy(lab2rgb(lab)).float().permute(0, 3, 1, 2)


def chroma(ab):
    return (ab * 128).pow(2).sum(dim=1).sqrt().mean().item()


L_train, ab_train = rgb_to_lab(x_train)
L_val, ab_val = rgb_to_lab(x_val)
L_test, ab_test = rgb_to_lab(x_test)

train_loader = DataLoader(TensorDataset(L_train, ab_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TensorDataset(L_val, ab_val), batch_size=BATCH_SIZE)
test_loader = DataLoader(TensorDataset(L_test, ab_test), batch_size=BATCH_SIZE)

gray_test = L_test                               # l'entrée noir et blanc
gray = (x_test * torch.tensor([0.299, 0.587, 0.114]).view(1, 3, 1, 1)).sum(dim=1, keepdim=True)
gray_copy_mse = F.mse_loss(gray.repeat(1, 3, 1, 1), x_test).item()
print("référence (noir et blanc recopié), MSE test :", round(gray_copy_mse, 5))
print("chroma des vraies images :", round(chroma(ab_test), 2))

## 4. Modèle 1 : un autoencodeur convolutionnel minuscule

Encodeur : une convolution (8 filtres) puis un max-pooling → un code de 16×16×4 (l'espace latent).
Décodeur : on réagrandit et une convolution sort les 2 canaux a et b. 446 paramètres en tout.

In [ ]:
class SmallColoriser(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(),   # 32x32x8
            nn.MaxPool2d(2),                            # 16x16x8
            nn.Conv2d(8, 4, 3, padding=1), nn.ReLU(),   # 16x16x4  <- espace latent
        )
        self.decoder = nn.Sequential(
            nn.Upsample(scale_factor=2),                # 32x32x4
            nn.Conv2d(4, 2, 3, padding=1), nn.Tanh(),   # 32x32x2 = a, b
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))


torch.manual_seed(SEED)
cae = SmallColoriser()
print("paramètres :", count_params(cae))
cae_history = train_model(cae, train_loader, val_loader)
torch.save(cae.state_dict(), "models/small_coloriser.pt")
plot_learning_curve(cae_history, "Autoencodeur, L -> ab")
ab_cae = predict(cae, L_test)
rgb_cae = lab_to_rgb(L_test, ab_cae)

## 5. Modèle 2 : un U-Net

Même idée encodeur-décodeur, mais plus profond, et chaque étage du décodeur reçoit aussi la sortie de l'étage
correspondant de l'encodeur (les « skip connections ») : les détails fins n'ont pas à passer par le goulot.

In [ ]:
def conv_block(c_in, c_out):
    return nn.Sequential(
        nn.Conv2d(c_in, c_out, 3, padding=1), nn.BatchNorm2d(c_out), nn.ReLU(),
        nn.Conv2d(c_out, c_out, 3, padding=1), nn.BatchNorm2d(c_out), nn.ReLU(),
    )


class UNetColoriser(nn.Module):
    def __init__(self, ch=(16, 32, 64)):
        super().__init__()
        self.enc1 = conv_block(1, ch[0])                       # 32x32
        self.enc2 = conv_block(ch[0], ch[1])                   # 16x16
        self.bottleneck = conv_block(ch[1], ch[2])             # 8x8
        self.up2 = nn.ConvTranspose2d(ch[2], ch[1], 2, stride=2)
        self.dec2 = conv_block(2 * ch[1], ch[1])               # 16x16 (agrandi + raccourci)
        self.up1 = nn.ConvTranspose2d(ch[1], ch[0], 2, stride=2)
        self.dec1 = conv_block(2 * ch[0], ch[0])               # 32x32
        self.out = nn.Conv2d(ch[0], 2, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(F.max_pool2d(e1, 2))
        b = self.bottleneck(F.max_pool2d(e2, 2))
        d2 = self.dec2(torch.cat([self.up2(b), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return torch.tanh(self.out(d1))


torch.manual_seed(SEED)
unet = UNetColoriser()
print("paramètres :", count_params(unet))
unet_history = train_model(unet, train_loader, val_loader)
torch.save(unet.state_dict(), "models/unet_coloriser.pt")
plot_learning_curve(unet_history, "U-Net, L -> ab")
ab_unet = predict(unet, L_test)
rgb_unet = lab_to_rgb(L_test, ab_unet)

## 6. Comparaison

In [ ]:
summary = pd.DataFrame({
    "modèle": ["noir et blanc recopié (référence)", "autoencodeur L -> ab", "U-Net L -> ab"],
    "paramètres": [0, count_params(cae), count_params(unet)],
    "MSE test (RGB)": [gray_copy_mse, F.mse_loss(rgb_cae, x_test).item(), F.mse_loss(rgb_unet, x_test).item()],
    "chroma moyenne": [0.0, chroma(ab_cae), chroma(ab_unet)],
}).set_index("modèle")
print("chroma des vraies images :", round(chroma(ab_test), 2))
display(summary.style.format({"MSE test (RGB)": "{:.5f}", "chroma moyenne": "{:.2f}"}))

idx = torch.arange(16, 26)
show_rows({"entrée noir et blanc": gray_test[idx], "autoencodeur": rgb_cae[idx], "U-Net": rgb_unet[idx],
           "vraie image": x_test[idx]}, title="Colorisation (images de test)")

## 7. Pour aller plus loin

- Les deux modèles colorisent trop timidement (chroma plus basse que les vraies images) : la MSE pousse vers des
  couleurs « moyennes ». Essaie une autre loss, ou de prédire des classes de couleur au lieu de valeurs.
- Fais varier la taille de l'espace latent de l'autoencodeur et regarde l'erreur bouger.
- Essaie sur tes propres photos (redimensionnées en 32×32, puis en plus grand).